In [2]:
pip install pandas matplotlib numpy

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime


In [4]:
df=pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Aminu_Assignment/Aminu_task1.xlsx",sheet_name="Cleaned Data")
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0


In [5]:
missing_check_rt_count=df['check_rt'].isna().sum()
print(missing_check_rt_count)
#missing_check_rt_count is non_null . Can't be reliably used for determining prdouct type

1


In [6]:
#product type determination can be done by checking category and produuct size : samples are for professionals generally
def product_type(row):
    cat = str(row['category']).lower()
    name = str(row['product_name']).lower()
    if cat == 'b2b' or cat == 'kit':
        return 'Professional'
    if 'wand' in name or 'equipment' in name:
        return 'Professional'
    if any(size in name for size in ['5ml', '10ml', 'sample']) and cat != 'b2b':
        return 'Professional (Sample)'
    retail_cats = ['cleanser', 'serum', 'oil', 'cream', 'masks & exfoliators', 'body', 'spf', 'combo']
    if cat in retail_cats:
        return 'Retail'
    return 'Retail'
df['product_type'] = df.apply(product_type, axis=1)
df.head(20)

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt,product_type
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0,Retail
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0,Retail
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0,Retail
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0,Retail
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0,Retail
5,2500,2025-11-01,A,FG - VitC Body Oil - 100ml,11,2025-11-12,Body,1144.07,0.0,0.36,8054.25,1449.76,9504.01,0.0,Retail
6,2500,2025-11-01,A,FG - Eye Serum - 10ml,19,2025-11-12,Serum,1567.80,0.0,0.36,19064.45,3431.60,22496.05,0.0,Professional (Sample)
7,2501,2025-11-04,B,FG - Milky Mineral Sunscreen - 5ml,100,2025-11-11,SPF,0.00,0.0,0.36,0.00,0.00,0.00,0.0,Professional (Sample)
8,2502,2025-11-05,C,FG - Milky Mineral Sunscreen - 5ml,70,2025-11-11,SPF,0.00,0.0,0.36,0.00,0.00,0.00,0.0,Professional (Sample)
9,2503,2025-11-05,D,FG - Milky Mineral Sunscreen - 5ml,70,2025-11-11,SPF,0.00,0.0,0.36,0.00,0.00,0.00,0.0,Professional (Sample)


In [16]:
#we can't use treat_disc and retail-disc as parameters for judging account_type as even retail account can get one time treat_disc as promotional offer
#we can't use appoint_date as that is dispatch date
#we can however use product-type and see if an account has both professional and retail type product in their order
acct_types = df.groupby('account_code')['product_type'].apply(set).to_dict() #creating sets of each account code and then convert it to dictionary with key value being account_code
def account_type(acct):
  types=acct_types.get(acct, set()) #get product type set for an account or initialise with empty set
  has_retail=any(t=='Retail' for t in types)
  has_professional=any('Professional' in t for t in types)
  if has_retail and has_professional:
    return 'Both'
  elif has_professional:
    return 'Professional'
  else:
    return 'Retail'
df['account_type']=df['account_code'].map(account_type)
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt,product_type,account_type
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0,Retail,Both
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0,Retail,Both
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0,Retail,Both
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0,Retail,Both
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0,Retail,Both


In [18]:
#order_status on basis of appointment_date
#fulfillment_stage , assuming any order with missing date is awaiting schedule , if appointment date has passed ,it's in warehouse .
today = pd.Timestamp.now().normalize()

def order_status(row):
    if pd.isna(row['appointment_date']):
        return 'No Appointment'
    elif row['appointment_date'] < today:
        return 'Overdue – Past Appointment'
    else:
        return 'Scheduled'

df['order_status'] = df.apply(order_status, axis=1)

def fulfillment_stage(row):
    if pd.isna(row['appointment_date']):
        return 'Awaiting Scheduling'
    elif row['appointment_date'] < today:
        return 'Overdue - In Warehouse'
    elif (row['appointment_date'] - today).days <= 3:
        return 'Picking Soon'
    else:
        return 'Planned'

df['fulfillment_stage'] = df.apply(fulfillment_stage, axis=1)

df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt,product_type,account_type,order_status,fulfillment_stage
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse


In [20]:
#assigning account manager on basis of account type . Since I don't have names, assigning them by team names
def manager_by_type(acc_type):
    if acc_type == 'Retail':
        return 'Retail Sales Team'
    elif acc_type == 'Professional':
        return 'B2B Sales Team'
    else:
        return 'Key Account Manager'

df['account_manager'] = df['account_type'].map(manager_by_type)
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt,product_type,account_type,order_status,fulfillment_stage,account_manager
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager


In [21]:
#assigning fulfillment teams by product_type and size / price
def fulfillment_owner(row):
    if row['product_type'] == 'Professional':
        return 'Team B (B2B)'
    elif row['quantity'] > 50 or row['final_price'] > 50000:
        return 'Team A (Bulk)'
    elif 'Sample' in row['product_type']:
        return 'Team C (Samples)'
    else:
        return 'Team D (Retail)'

df['fulfillment_owner'] = df.apply(fulfillment_owner, axis=1)
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,base_price,gst,final_price,check_rt,product_type,account_type,order_status,fulfillment_stage,account_manager,fulfillment_owner
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,786.44,141.56,928.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail)
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,1437.29,258.71,1696.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail)
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,1925.42,346.58,2272.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail)
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,650.85,117.15,768.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail)
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,7484.76,1347.26,8832.02,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail)


In [22]:
#prioritizing orders with upcoming appoint date  , followed by high value , followed by value , followed by quantity
today = pd.Timestamp.now().normalize()
def priority_flag(row):
    if pd.notna(row['appointment_date']):
        days_to = (row['appointment_date'] - today).days
        if 0 <= days_to <= 3:
            return 'High – Urgent'
    if row['final_price'] > 800000:
        return 'High – High Value'
    if row['quantity'] > 100:
        return 'High – Bulk'
    if pd.notna(row['appointment_date']) and row['appointment_date'] < today:
        return 'High – Past Due'
    return 'Normal'

df['priority_flag'] = df.apply(priority_flag, axis=1)
df.head()


,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,...,gst,final_price,check_rt,product_type,account_type,order_status,fulfillment_stage,account_manager,fulfillment_owner,priority_flag
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,...,141.56,928.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,...,258.71,1696.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,...,346.58,2272.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,...,117.15,768.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,...,1347.26,8832.02,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due


In [23]:
#assuming dispatch date should be two days before appoint date ideally and if order_date is two days before appoint-date , then it should be one day ahead
def expected_dispatch(row):
    if pd.notna(row['appointment_date']):
        dispatch = row['appointment_date'] - pd.Timedelta(days=2)
        # Ensure dispatch is not before order date
        if dispatch < row['date']:
            dispatch = row['date'] + pd.Timedelta(days=1)
        return dispatch
    else:
        return pd.NaT
df['expected_dispatch_date'] = df.apply(expected_dispatch, axis=1)
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,...,final_price,check_rt,product_type,account_type,order_status,fulfillment_stage,account_manager,fulfillment_owner,priority_flag,expected_dispatch_date
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,...,928.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,...,1696.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,...,2272.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,...,768.00,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,...,8832.02,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10


In [27]:
#Writing internal notes about missing appoint date , sample products , 0 quantity rows  , priority flag info and check r/t flag status.
def internal_notes(row):
    notes = []
    if pd.isna(row['appointment_date']):
        notes.append('No appointment date')
    if row['quantity'] == 0:
        notes.append('Zero quantity')
    if row['unit_price'] == 0 and row['quantity'] > 0:
        notes.append('Free sample')
    if row['priority_flag'] != 'Normal':
        notes.append(f"Priority: {row['priority_flag']}")
    if pd.isna(row['check_rt']) or row['check_rt'] == 0:  # check_rt is False or nane
        notes.append('R/T not verified')
    return ' | '.join(notes) if notes else 'Normal'
df['internal_notes'] = df.apply(internal_notes, axis=1)
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,...,check_rt,product_type,account_type,order_status,fulfillment_stage,account_manager,fulfillment_owner,priority_flag,expected_dispatch_date,internal_notes
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,...,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,...,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,...,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,...,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,...,0.0,Retail,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified


In [30]:
#days to appointment
today = pd.Timestamp.now().normalize()
df['days_until_appointment'] = (df['appointment_date'] - today).dt.days
df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,...,account_type,order_status,fulfillment_stage,account_manager,fulfillment_owner,priority_flag,expected_dispatch_date,internal_notes,days_to_appointment,days_until_appointment
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,...,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,...,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,...,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,...,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,...,Both,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0


In [32]:
#Month where order was placed for trend analysis
df['order_month'] = df['date'].dt.month_name()

df.head()

,order_id,date,account_code,product_name,quantity,appointment_date,category,unit_price,treat_disc,retail_disc,...,order_status,fulfillment_stage,account_manager,fulfillment_owner,priority_flag,expected_dispatch_date,internal_notes,days_to_appointment,days_until_appointment,order_month
0,2500,2025-11-01,A,FG - AHA Face Wash - 100ml,1,2025-11-12,Cleanser,1228.81,0.0,0.36,...,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0,November
1,2500,2025-11-01,A,FG - Sea Salt Body Scrub - 200ml,1,2025-11-12,Body,2245.76,0.0,0.36,...,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0,November
2,2500,2025-11-01,A,FG - Radiance Face Oil - 30ml,1,2025-11-12,Oil,3008.47,0.0,0.36,...,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0,November
3,2500,2025-11-01,A,FG - Melting Balm Cleanser - 50ml,1,2025-11-12,Cleanser,1016.95,0.0,0.36,...,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0,November
4,2500,2025-11-01,A,FG - Copper Peptide Serum - 30ml,3,2025-11-12,Serum,3898.31,0.0,0.36,...,Overdue – Past Appointment,Overdue - In Warehouse,Key Account Manager,Team D (Retail),High – Past Due,2025-11-10,Priority: High – Past Due | R/T not verified,11.0,-116.0,November


In [34]:
df.to_excel('/content/drive/MyDrive/Colab Notebooks/Aminu_Assignment/Aminu_task2.xlsx', index=False)
print("Completed Task 2")

Completed Task 2
